# 02 — ASR baseline: PhoWhisper (VI) + Whisper-small (EN/ZH/KO)

Tests the ASR stage per CLAUDE.md's model stack baseline. Unlike MT
(`03_mt.ipynb`), **no ONNX export happens here** — pre-converted ONNX
already exists for both models and is downloaded and used as-is:

- VI: [`huuquyet/PhoWhisper-base`](https://huggingface.co/huuquyet/PhoWhisper-base)
  — ONNX conversion of `vinai/PhoWhisper-base` (BSD-3-Clause, the
  CLAUDE.md baseline). Ships encoder/decoder in fp32 *and* pre-quantized
  (int8) variants.
- EN/ZH/KO: [`onnx-community/whisper-small`](https://huggingface.co/onnx-community/whisper-small)
  — ONNX conversion of `openai/whisper-small` (MIT), multilingual, one
  model covers all three per CLAUDE.md's model stack table. Also ships
  fp32 and multiple quantized variants (int8/uint8/q4/fp16/bnb4).

This notebook: loads each pre-made ONNX via `optimum`, validates output
against a PyTorch reference and against ground-truth transcripts (WER),
compares the fp32 vs. int8-quantized variant, and packages the chosen
artifact into `models/asr/<lang>/`.


In [1]:
from pathlib import Path

VI_REPO = "huuquyet/PhoWhisper-base"
MULTI_REPO = "onnx-community/whisper-small"
MULTI_PT_REPO = "openai/whisper-small"  # onnx-community/whisper-small only ships ONNX weights, not PyTorch
VI_FINAL_DIR = Path("../models/asr/vi")
MULTI_FINAL_DIR = Path("../models/asr/en-zh-ko")


## 1. VI: PhoWhisper-base — download pre-made ONNX (fp32), no conversion

In [2]:
from huggingface_hub import snapshot_download

vi_fp32_dir = snapshot_download(
    repo_id=VI_REPO,
    allow_patterns=[
        "onnx/encoder_model.onnx",
        "onnx/decoder_model_merged.onnx",
        "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
        "config.json", "generation_config.json", "preprocessor_config.json",
        "vocab.json", "merges.txt", "normalizer.json", "added_tokens.json",
    ],
)
vi_fp32_dir


/opt/homebrew/Caskroom/miniconda/base/envs/audio-lab/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 120989.54it/s]

'/Users/minhld/.cache/huggingface/hub/models--huuquyet--PhoWhisper-base/snapshots/7623a803636be73f54c30dfa4ce1c42cdc9bbbcf'

## 2. VI test set: VIVOS (via a Parquet mirror -- the original `vivos` loading script is no longer supported by `datasets`)

In [3]:
from datasets import load_dataset

vivos = load_dataset("quocanh34/viet_vivos", split="test", streaming=True)
vi_samples = [s for _, s in zip(range(10), vivos)]
[s["transcription"] for s in vi_samples]


['nửa vòng trái đất hơn bảy năm',
 'bốn chiếc trống đại phèng la',
 'của những thế lực hùng mạnh',
 'thế nhưng khi giá phôi thép thế giới cao dần',
 'tên gọi tại địa phương là trắc thối',
 'doanh nghiệp tuyển dụng lao động nước ngoài',
 'tức khắc ập về kéo mọi người thứ trở lại thời hoang sơ',
 'mới đây nhất là các tiểu vương quốc ả rập thống nhất',
 'lần lữa mãi mới gặp được tuyên',
 'tôi nghĩ thủ tướng sẽ không né tránh bất cứ câu hỏi nào']

## 3. PyTorch reference transcription (VI)

In [4]:
import jiwer
from transformers import WhisperForConditionalGeneration, WhisperProcessor

vi_processor = WhisperProcessor.from_pretrained(VI_REPO)
vi_pt_model = WhisperForConditionalGeneration.from_pretrained(VI_REPO)
vi_pt_model.eval()


def transcribe_pt(model, processor, samples, language):
    hyps = []
    for s in samples:
        audio = s["audio"]
        inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt")
        gen = model.generate(**inputs, language=language, task="transcribe", max_new_tokens=64)
        hyps.append(processor.batch_decode(gen, skip_special_tokens=True)[0].strip().rstrip("."))
    return hyps


vi_refs = [s["transcription"] for s in vi_samples]
vi_pt_hyps = transcribe_pt(vi_pt_model, vi_processor, vi_samples, "vi")
for ref, hyp in zip(vi_refs, vi_pt_hyps):
    print(f"ref: {ref}\nhyp: {hyp}\n")
print("PyTorch WER (VI, PhoWhisper-base):", jiwer.wer(vi_refs, vi_pt_hyps))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


ref: nửa vòng trái đất hơn bảy năm
hyp: nửa vòng trái đất hơn bảy năm

ref: bốn chiếc trống đại phèng la
hyp: bốn chiếc trống đại phàn la

ref: của những thế lực hùng mạnh
hyp: của những thế lực hùng mạnh

ref: thế nhưng khi giá phôi thép thế giới cao dần
hyp: thế nhưng khi giá phôi thép thế giới cao dần

ref: tên gọi tại địa phương là trắc thối
hyp: tên gọi tại địa phương là trắc thối

ref: doanh nghiệp tuyển dụng lao động nước ngoài
hyp: doanh nghiệp tuyển dụng lao động nước ngoài

ref: tức khắc ập về kéo mọi người thứ trở lại thời hoang sơ
hyp: tức khắc ập về kéo mọi người thứ trở lại thời hoang sơ

ref: mới đây nhất là các tiểu vương quốc ả rập thống nhất
hyp: mới đây nhất là các tiểu vương quốc ả rập thống nhất

ref: lần lữa mãi mới gặp được tuyên
hyp: lần nữa mãi mới gặp được tuyên

ref: tôi nghĩ thủ tướng sẽ không né tránh bất cứ câu hỏi nào
hyp: tôi nghĩ thủ tướng sẽ không né tránh bất cứ câu hỏi nào

PyTorch WER (VI, PhoWhisper-base): 0.022222222222222223


## 4. ONNX Runtime transcription (VI, fp32) -- verify it matches the PyTorch reference

In [5]:
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq


def transcribe_ort(model, processor, samples, language):
    hyps = []
    for s in samples:
        audio = s["audio"]
        inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt")
        gen = model.generate(**inputs, language=language, task="transcribe", max_new_tokens=64)
        hyps.append(processor.batch_decode(gen, skip_special_tokens=True)[0].strip().rstrip("."))
    return hyps


vi_ort_model = ORTModelForSpeechSeq2Seq.from_pretrained(
    vi_fp32_dir, use_cache=True, use_merged=True, provider="CPUExecutionProvider"
)
vi_ort_hyps = transcribe_ort(vi_ort_model, vi_processor, vi_samples, "vi")
matches = sum(a == b for a, b in zip(vi_pt_hyps, vi_ort_hyps))
print(f"ONNX (fp32) matches PyTorch on {matches}/{len(vi_samples)} samples")
print("ONNX (fp32) WER (VI):", jiwer.wer(vi_refs, vi_ort_hyps))
assert matches == len(vi_samples), "pre-made ONNX diverged from the PyTorch reference"


Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.


ONNX (fp32) matches PyTorch on 10/10 samples
ONNX (fp32) WER (VI): 0.022222222222222223


## 5. Quantized (int8) ONNX -- quality vs. size tradeoff

`optimum`'s file auto-discovery gets confused when both fp32 and
quantized ONNX files sit in the same directory and a `decoder_file_name`
is passed explicitly (it then also goes looking for a separate
`decoder_with_past` file that doesn't exist here, since this repo only
ships the merged decoder). Downloading the quantized files into their
own directory sidesteps this -- default auto-discovery unambiguously
picks them up.


In [6]:
import shutil
from huggingface_hub import hf_hub_download

vi_int8_dir = Path("../models/asr/_vi_int8_staging")
(vi_int8_dir / "onnx").mkdir(parents=True, exist_ok=True)

quant_files = [
    "onnx/encoder_model_quantized.onnx",
    "onnx/decoder_model_merged_quantized.onnx",
    "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
    "config.json", "generation_config.json", "preprocessor_config.json",
    "vocab.json", "merges.txt", "normalizer.json", "added_tokens.json",
]
for f in quant_files:
    src = hf_hub_download(VI_REPO, f)
    dest = vi_int8_dir / f
    if not dest.exists():
        shutil.copy(src, dest)

vi_int8_model = ORTModelForSpeechSeq2Seq.from_pretrained(
    str(vi_int8_dir), use_cache=True, use_merged=True, provider="CPUExecutionProvider"
)
vi_int8_hyps = transcribe_ort(vi_int8_model, vi_processor, vi_samples, "vi")
for ref, hyp in zip(vi_refs, vi_int8_hyps):
    print(f"ref: {ref}\nhyp: {hyp}\n")
print("ONNX (int8) WER (VI):", jiwer.wer(vi_refs, vi_int8_hyps))

fp32_size = sum((Path(vi_fp32_dir) / f).stat().st_size for f in ["onnx/encoder_model.onnx", "onnx/decoder_model_merged.onnx"])
int8_size = sum((vi_int8_dir / f).stat().st_size for f in ["onnx/encoder_model_quantized.onnx", "onnx/decoder_model_merged_quantized.onnx"])
print(f"fp32 size: {fp32_size / 1e6:.1f} MB, int8 size: {int8_size / 1e6:.1f} MB ({fp32_size / int8_size:.1f}x smaller)")


Could not find any ONNX files with standard file name encoder_model.onnx, files found: [PosixPath('onnx/encoder_model_quantized.onnx'), PosixPath('onnx/decoder_model_merged_quantized.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.


Could not find any ONNX files with standard file name decoder_model_merged.onnx, files found: [PosixPath('onnx/encoder_model_quantized.onnx'), PosixPath('onnx/decoder_model_merged_quantized.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.


Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.


ref: nửa vòng trái đất hơn bảy năm
hyp: nửa vòng trái đất hơn bảy năm

ref: bốn chiếc trống đại phèng la
hyp: bốn chiếc trống đại phàn la

ref: của những thế lực hùng mạnh
hyp: của những thế lực hùng mạnh

ref: thế nhưng khi giá phôi thép thế giới cao dần
hyp: thế nhưng khi giá phôi thép thế giới cao dần

ref: tên gọi tại địa phương là trắc thối
hyp: tên gọi tại địa phương là trắc thối

ref: doanh nghiệp tuyển dụng lao động nước ngoài
hyp: doanh nghiệp tuyển dụng lao động nước ngoài

ref: tức khắc ập về kéo mọi người thứ trở lại thời hoang sơ
hyp: tức khắc ập về kéo mọi người thứ trở lại thời hoang sơ

ref: mới đây nhất là các tiểu vương quốc ả rập thống nhất
hyp: mới đây nhất là các tiểu vương quốc ả rập thống nhất

ref: lần lữa mãi mới gặp được tuyên
hyp: lần nữa mãi mới gặp được tuyên

ref: tôi nghĩ thủ tướng sẽ không né tránh bất cứ câu hỏi nào
hyp: tôi nghĩ thủ tướng sẽ không né tránh bất cứ câu hỏi nào

ONNX (int8) WER (VI): 0.022222222222222223
fp32 size: 291.0 MB, int8 size: 76

On this 10-sample slice, int8 quantization costs no measurable quality
(same WER, same errors) for ~3.8x less disk/memory -- well within the
~300MB/pair-ish budget headroom implied by CLAUDE.md's ~2GB total memory
target. Packaging the **int8 variant** as the real artifact below rather
than fp32 (unlike MT, where fp32 was kept as the baseline and
quantization left as follow-up work) -- there's no reason not to here
since it's already validated and already exists.


## 6. Package VI artifacts into `models/asr/vi/`

In [7]:
from datetime import date

VI_FINAL_DIR.mkdir(parents=True, exist_ok=True)
shutil.copyfile(vi_int8_dir / "onnx" / "encoder_model_quantized.onnx", VI_FINAL_DIR / "encoder_model.onnx")
shutil.copyfile(vi_int8_dir / "onnx" / "decoder_model_merged_quantized.onnx", VI_FINAL_DIR / "decoder_model_merged.onnx")
shutil.copyfile(vi_int8_dir / "tokenizer.json", VI_FINAL_DIR / "tokenizer.json")
shutil.copyfile(vi_int8_dir / "preprocessor_config.json", VI_FINAL_DIR / "preprocessor_config.json")
shutil.copyfile(vi_int8_dir / "generation_config.json", VI_FINAL_DIR / "generation_config.json")

manifest = f"""# asr/vi

- Source: {VI_REPO} (pre-converted ONNX -- not exported by us; underlying
  weights are vinai/PhoWhisper-base, BSD-3-Clause)
- Packaged: {date.today().isoformat()}
- Precision: int8 (quantized variant from the source repo, not quantized
  by us). On a 10-sample VIVOS slice, identical WER to fp32
  ({jiwer.wer(vi_refs, vi_int8_hyps):.3f}) at ~3.8x smaller
  ({int8_size / 1e6:.0f}MB vs. {fp32_size / 1e6:.0f}MB) -- see
  experiments/02_asr.ipynb section 5.
- Decode: greedy (beam=1) via `model.generate(num_beams=1)` default,
  language/task forced to vi/transcribe
- Eval: WER against 10 VIVOS test-set samples (quocanh34/viet_vivos
  mirror -- the original `vivos` HF dataset uses a loading script no
  longer supported by recent `datasets` versions)
- Preprocessing note: needs a log-mel spectrogram feature extractor
  (`preprocessor_config.json` has the params) *before* the ONNX
  encoder -- unlike MT's plain tokenizer, this isn't just a lookup
  table, and isn't part of the ONNX graph. Not yet ported to Rust; see
  docs/TODO.md.

Files:
- `encoder_model.onnx`, `decoder_model_merged.onnx` (int8 quantized)
- `tokenizer.json` (already a proper fast tokenizer -- Whisper's BPE
  tokenizer has a native fast/tokenizer.json variant, unlike Marian's
  sentencepiece tokenizer in the MT stage; no manual conversion needed)
- `preprocessor_config.json` (feature extractor params: n_mels, hop
  length, sampling rate, etc.)
- `generation_config.json` (decoder_start_token_id, forced language/task
  token ids, suppress_tokens list)
"""
(VI_FINAL_DIR / "MANIFEST.md").write_text(manifest)

shutil.rmtree(vi_int8_dir)
sorted(p.name for p in VI_FINAL_DIR.iterdir())


['MANIFEST.md',
 'decoder_model_merged.onnx',
 'encoder_model.onnx',
 'generation_config.json',
 'preprocessor_config.json',
 'tokenizer.json']

## 7. EN/ZH/KO: generic Whisper-small — same pattern, lighter check

CLAUDE.md's baseline for EN/ZH/KO is one shared `Whisper-Small` (as
opposed to per-language models) since Whisper is natively multilingual.
This section validates the EN path only (LibriSpeech dummy set is the
standard quick sanity-check corpus for this) -- ZH/KO aren't tested here
since it's the same artifact and the interesting validation (does the
pre-made ONNX match PyTorch, does quantization hurt) is language-
independent. Tracked as a follow-up in docs/TODO.md.


In [8]:
multi_int8_dir = Path("../models/asr/_en_int8_staging")
(multi_int8_dir / "onnx").mkdir(parents=True, exist_ok=True)

multi_files = [
    "onnx/encoder_model_quantized.onnx",
    "onnx/decoder_model_merged_quantized.onnx",
    "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
    "config.json", "generation_config.json", "preprocessor_config.json",
    "vocab.json", "merges.txt", "normalizer.json", "added_tokens.json",
]
for f in multi_files:
    src = hf_hub_download(MULTI_REPO, f)
    dest = multi_int8_dir / f
    if not dest.exists():
        shutil.copy(src, dest)

en_ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
en_samples = [en_ds[i] for i in range(5)]
en_refs = [s["text"].lower().rstrip(".") for s in en_samples]

multi_processor = WhisperProcessor.from_pretrained(MULTI_PT_REPO)
multi_pt_model = WhisperForConditionalGeneration.from_pretrained(MULTI_PT_REPO)
multi_pt_model.eval()
multi_ort_model = ORTModelForSpeechSeq2Seq.from_pretrained(
    str(multi_int8_dir), use_cache=True, use_merged=True, provider="CPUExecutionProvider"
)


def norm(s):
    return s.lower().strip().rstrip(".")


en_pt_hyps = [norm(h) for h in transcribe_pt(multi_pt_model, multi_processor, en_samples, "en")]
en_ort_hyps = [norm(h) for h in transcribe_ort(multi_ort_model, multi_processor, en_samples, "en")]

for ref, pt_hyp, ort_hyp in zip(en_refs, en_pt_hyps, en_ort_hyps):
    print(f"ref:      {ref}\npt fp32:  {pt_hyp}\nonnx int8: {ort_hyp}\n")

print("PyTorch (fp32) WER (EN):", jiwer.wer(en_refs, en_pt_hyps))
print("ONNX (int8) WER (EN):", jiwer.wer(en_refs, en_ort_hyps))


Could not find any ONNX files with standard file name encoder_model.onnx, files found: [PosixPath('onnx/encoder_model_quantized.onnx'), PosixPath('onnx/decoder_model_merged_quantized.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.


Could not find any ONNX files with standard file name decoder_model_merged.onnx, files found: [PosixPath('onnx/encoder_model_quantized.onnx'), PosixPath('onnx/decoder_model_merged_quantized.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.


Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.


ref:      mister quilter is the apostle of the middle classes and we are glad to welcome his gospel
pt fp32:  mr. quilter is the apostle of the middle classes, and we are glad to welcome his gospel
onnx int8: mr. quilter is the apostle of the middle classes, and we are glad to welcome his gospel

ref:      nor is mister quilter's manner less interesting than his matter
pt fp32:  nor is mr. quilter's manner less interesting than his matter
onnx int8: nor is mr. quilter's manner less interesting than his matter

ref:      he tells us that at this festive season of the year with christmas and roast beef looming before us similes drawn from eating and its results occur most readily to the mind
pt fp32:  he tells us that at this festive season of the year, with christmas and roast beef looming before us, symbolies drawn from eating and its results occur most readily to the mind
onnx int8: he tells us that at this festive season of the year, with christmas and roast beef looming before us, s

## 8. Package EN/ZH/KO artifacts into `models/asr/en-zh-ko/`

In [9]:
MULTI_FINAL_DIR.mkdir(parents=True, exist_ok=True)
shutil.copyfile(multi_int8_dir / "onnx" / "encoder_model_quantized.onnx", MULTI_FINAL_DIR / "encoder_model.onnx")
shutil.copyfile(multi_int8_dir / "onnx" / "decoder_model_merged_quantized.onnx", MULTI_FINAL_DIR / "decoder_model_merged.onnx")
shutil.copyfile(multi_int8_dir / "tokenizer.json", MULTI_FINAL_DIR / "tokenizer.json")
shutil.copyfile(multi_int8_dir / "preprocessor_config.json", MULTI_FINAL_DIR / "preprocessor_config.json")
shutil.copyfile(multi_int8_dir / "generation_config.json", MULTI_FINAL_DIR / "generation_config.json")

manifest = f"""# asr/en-zh-ko

- Source: {MULTI_REPO} (pre-converted ONNX -- not exported by us;
  underlying weights are openai/whisper-small, MIT)
- Packaged: {date.today().isoformat()}
- Precision: int8 (quantized variant from the source repo)
- One shared multilingual model for EN/ZH/KO per CLAUDE.md's model
  stack table -- language is selected at generate() time via the
  `language=` kwarg, not by swapping models.
- Eval: only the EN path validated so far (5 LibriSpeech-dummy samples,
  WER logged in experiments/02_asr.ipynb section 7). ZH/KO not tested
  yet -- same artifact, tracked as a follow-up in docs/TODO.md.
- Same preprocessing note as asr/vi/MANIFEST.md (needs a log-mel
  feature extractor ahead of the ONNX encoder).

Files: same shape as asr/vi/ (encoder_model.onnx, decoder_model_merged.onnx,
tokenizer.json, preprocessor_config.json, generation_config.json).
"""
(MULTI_FINAL_DIR / "MANIFEST.md").write_text(manifest)

shutil.rmtree(multi_int8_dir)
sorted(p.name for p in MULTI_FINAL_DIR.iterdir())


['MANIFEST.md',
 'decoder_model_merged.onnx',
 'encoder_model.onnx',
 'generation_config.json',
 'preprocessor_config.json',
 'tokenizer.json']